In [ ]:
PROJECT_ROOT = "/content/drive/MyDrive/CLASS_AlignED_MVP"

In [ ]:
# NB2 - Cell 2: Install Gemini SDK
!pip -q install -U google-genai

In [ ]:
# NB2 - Cell 3: Set keys (use Colab Secrets if possible)
import os, json
PROJECT_ROOT = "/content/drive/MyDrive/CLASS_AlignED_MVP"
MANIFEST = f"{PROJECT_ROOT}/processed/manifest.json"
OUT_EXTRACTED = f"{PROJECT_ROOT}/processed/extracted"
os.makedirs(OUT_EXTRACTED, exist_ok=True)

# Option 1: set directly (quick test)
# os.environ["GEMINI_API_KEY"] = "YOUR_GEMINI_KEY"

# Option 2: in Colab, set in "Secrets" UI, then read:
# os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

In [ ]:
# NB2 - Cell 4: Gemini client + prompt template
from google import genai

client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))

SYSTEM_INSTRUCTIONS = """You extract structured information from university course syllabi.
Return strict JSON that matches the schema. Do not include markdown.
All extracted facts must include supporting evidence as chunk_id references."""

SCHEMA = {
  "doc_id": "string",
  "course": {"title":"string","code":"string","term":"string"},
  "learning_outcomes":[{"id":"string","text":"string","evidence":["chunk_id"]}],
  "assessments":[{"id":"string","name":"string","weight_percent":0,"evidence":["chunk_id"]}],
  "policies":[{"id":"string","type":"string","summary":"string","evidence":["chunk_id"]}],
  "tools":[{"id":"string","name":"string","evidence":["chunk_id"]}],
  "relations":[
    {"source_id":"string","target_id":"string","relation":"string","rationale":"string","evidence":["chunk_id"]}
  ],
  "recommendations":[
    {"id":"string","title":"string","description":"string",
     "supports":["learning_outcome_id"],
     "constrained_by":["PolicyPrinciple"],
     "risks":["string"],
     "mitigations":["string"],
     "evidence":["chunk_id"]}
  ]
}

PROMPT_TEMPLATE = """{system}

You are given syllabus chunks as JSONL records (each has chunk_id, section, text).
TASK:
1) Extract course metadata, learning outcomes, assessments, tools, and syllabus policies.
2) Create relations:
   - COURSE_HAS_OUTCOME (course -> outcome)
   - COURSE_HAS_ASSESSMENT (course -> assessment)
   - COURSE_REQUIRES_TOOL (course -> tool)
   - COURSE_HAS_POLICY (course -> policy)
   - ASSESSMENT_MEASURES_OUTCOME (assessment -> outcome) when reasonable
3) Produce 5-10 recommendations for responsible AI use in lesson planning/classroom/assessment design.
   Tag each recommendation with PolicyPrinciples (Accountability, Transparency, Privacy, Integrity, FacultyAuthority).
4) Every item must cite evidence chunk_id(s).

Return ONLY strict JSON matching this schema:
{schema}

INPUT CHUNKS:
{chunks}
"""

In [ ]:
# NB2 - Cell 4: Gemini client + prompt template
from google import genai

client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))

SYSTEM_INSTRUCTIONS = """You extract structured information from university course syllabi.
Return strict JSON that matches the schema. Do not include markdown.
All extracted facts must include supporting evidence as chunk_id references."""

SCHEMA = {
  "doc_id": "string",
  "course": {"title":"string","code":"string","term":"string"},
  "learning_outcomes":[{"id":"string","text":"string","evidence":["chunk_id"]}],
  "assessments":[{"id":"string","name":"string","weight_percent":0,"evidence":["chunk_id"]}],
  "policies":[{"id":"string","type":"string","summary":"string","evidence":["chunk_id"]}],
  "tools":[{"id":"string","name":"string","evidence":["chunk_id"]}],
  "relations":[
    {"source_id":"string","target_id":"string","relation":"string","rationale":"string","evidence":["chunk_id"]}
  ],
  "recommendations":[
    {"id":"string","title":"string","description":"string",
     "supports":["learning_outcome_id"],
     "constrained_by":["PolicyPrinciple"],
     "risks":["string"],
     "mitigations":["string"],
     "evidence":["chunk_id"]}
  ]
}

PROMPT_TEMPLATE = """{system}

You are given syllabus chunks as JSONL records (each has chunk_id, section, text).
TASK:
1) Extract course metadata, learning outcomes, assessments, tools, and syllabus policies.
2) Create relations:
   - COURSE_HAS_OUTCOME (course -> outcome)
   - COURSE_HAS_ASSESSMENT (course -> assessment)
   - COURSE_REQUIRES_TOOL (course -> tool)
   - COURSE_HAS_POLICY (course -> policy)
   - ASSESSMENT_MEASURES_OUTCOME (assessment -> outcome) when reasonable
3) Produce 5-10 recommendations for responsible AI use in lesson planning/classroom/assessment design.
   Tag each recommendation with PolicyPrinciples (Accountability, Transparency, Privacy, Integrity, FacultyAuthority).
4) Every item must cite evidence chunk_id(s).

Return ONLY strict JSON matching this schema:
{schema}

INPUT CHUNKS:
{chunks}
"""

In [ ]:
# NB2 - Cell 5: Load chunks and call Gemini (per syllabus)
from pathlib import Path

with open(MANIFEST, "r", encoding="utf-8") as r:
    manifest = json.load(r)

def load_jsonl(path: str, max_chunks=60):
    rows = []
    with open(path, "r", encoding="utf-8") as r:
        for i, line in enumerate(r):
            if i >= max_chunks: break
            rows.append(json.loads(line))
    return rows

def run_extraction(doc):
    chunks = load_jsonl(doc["chunks_jsonl"], max_chunks=80)  # increase if needed
    prompt = PROMPT_TEMPLATE.format(
        system=SYSTEM_INSTRUCTIONS,
        schema=json.dumps(SCHEMA, indent=2),
        chunks=json.dumps(chunks, ensure_ascii=False)
    )
    resp = client.models.generate_content(
        model="gemini-2.0-flash",  # fast & cheap for extraction; swap as needed
        contents=prompt
    )
    # Expect JSON in text
    text = resp.text.strip()
    data = json.loads(text)
    return data

for doc in manifest:
    data = run_extraction(doc)
    out_path = f"{OUT_EXTRACTED}/{doc['doc_id']}_extracted.json"
    with open(out_path, "w", encoding="utf-8") as w:
        json.dump(data, w, ensure_ascii=False, indent=2)
    print("Wrote:", out_path)